In [2]:
# Import required libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'matplotlib'

In [31]:
# Read the dataset
path = '../data/raw/dirty_dataset.csv'
df = pd.read_csv(path)
print('Shape:', df.shape)


Shape: (500000, 46)


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  str    
 1   Source                 500000 non-null  str    
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  str    
 4   End_Time               500000 non-null  str    
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                279623 non-null  float64
 8   End_Lng                279623 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            499999 non-null  str    
 11  Street                 499309 non-null  str    
 12  City                   499981 non-null  str    
 13  County                 500000 non-null  str    
 14  State                  500000 non-null  str    

<p> There are features of worng data types (Start_Time), and other features will many missied values (End_lat)</p>

In [33]:
df.iloc[:,-26:].info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 26 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Temperature(F)         489534 non-null  float64
 1   Wind_Chill(F)          370983 non-null  float64
 2   Humidity(%)            488870 non-null  float64
 3   Pressure(in)           491072 non-null  float64
 4   Visibility(mi)         488709 non-null  float64
 5   Wind_Direction         488803 non-null  str    
 6   Wind_Speed(mph)        463013 non-null  float64
 7   Precipitation(in)      357384 non-null  float64
 8   Weather_Condition      488899 non-null  str    
 9   Amenity                500000 non-null  bool   
 10  Bump                   500000 non-null  bool   
 11  Crossing               500000 non-null  bool   
 12  Give_Way               500000 non-null  bool   
 13  Junction               500000 non-null  bool   
 14  No_Exit                500000 non-null  bool   

<p>Explore the middle part of the data: missing values for important feature(Precipitation(in))</p>

In [34]:
df.iloc[:,-6:].info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   Traffic_Signal         500000 non-null  bool 
 1   Turning_Loop           500000 non-null  bool 
 2   Sunrise_Sunset         498517 non-null  str  
 3   Civil_Twilight         498517 non-null  str  
 4   Nautical_Twilight      498517 non-null  str  
 5   Astronomical_Twilight  498517 non-null  str  
dtypes: bool(2), str(4)
memory usage: 16.2 MB


<p>The End of the features: include some useless features that won't affect the model </p>

In [35]:
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
1,A-4694324,Source1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,...,False,False,False,False,False,False,Night,Night,Night,Night
2,A-5006183,Source1,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,...,False,False,False,False,True,False,Day,Day,Day,Day
3,A-4237356,Source1,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.680592,-92.993317,43.680574,-92.972223,1.054,...,False,False,False,False,False,False,Day,Day,Day,Day
4,A-6690583,Source1,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.395484,-118.985176,35.395476,-118.985995,0.046,...,False,False,False,False,False,False,Night,Night,Night,Night


<p> Get sense of how the data looks like</p>

In [36]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed')
df['End_Time'] = pd.to_datetime(df['End_Time'], format='mixed')
df['Weather_Timestamp'] = pd.to_datetime(df['Weather_Timestamp'],format='mixed')

print(df[['Start_Time','Weather_Timestamp']].dtypes)
print("Mean difference between 'Start_Time' and 'Weather_Timestamp': ", 
(df['Weather_Timestamp'] - df['Start_Time']).mean())

Start_Time           datetime64[ns]
Weather_Timestamp    datetime64[us]
dtype: object
Mean difference between 'Start_Time' and 'Weather_Timestamp':  0 days 00:00:34.307241543


<p> Data type casting to the suitable types, Here (Start time) will be enough for further exploration and engineering </p>

In [37]:
cat_names = [ 'Source','Country', 'Timezone', 'Amenity', 'Bump', 'Crossing', 
             'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 
             'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 
             'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']
print("Unique count of categorical features:")
for i in cat_names:
  print(i,df[i].unique().size)

Unique count of categorical features:
Source 3
Country 1
Timezone 5
Amenity 2
Bump 2
Crossing 2
Give_Way 2
Junction 2
No_Exit 2
Railway 2
Roundabout 2
Station 2
Stop 2
Traffic_Calming 2
Traffic_Signal 2
Turning_Loop 1
Sunrise_Sunset 3
Civil_Twilight 3
Nautical_Twilight 3
Astronomical_Twilight 3


<p> Categorical data unique values: feature with one unique value will be dropped (Country)</p>

In [38]:
# Feature that won't affect model

df = df.drop([
        "ID",
        "Source",
        "End_Lat",
        "End_Lng",
        "End_Time",
        "Distance(mi)",
        "Description",
        "City",
        "County",
        "State",
        "Zipcode",
        "Country",
        "Timezone",
        "Weather_Timestamp",
        "Wind_Chill(F)",
        "Wind_Direction",
        "Civil_Twilight",
        "Nautical_Twilight",
        "Astronomical_Twilight",
        "Turning_Loop",
    ], axis=1)
df.shape

(500000, 26)

<p> Clean the Data set from unwanted features</p>

In [39]:
missing = pd.DataFrame(df.isnull().sum()).reset_index()
missing.columns = ['Feature', 'Missing_Percent(%)']
missing['Missing_Percent(%)'] = missing['Missing_Percent(%)'].transform(lambda x: x / df.shape[0] * 100)
missing.loc[missing['Missing_Percent(%)']>0,:]

,Feature,Missing_Percent(%)
4,Street,0.1382
5,Airport_Code,0.2892
6,Temperature(F),2.0932
7,Humidity(%),2.2260
8,Pressure(in),1.7856
9,Visibility(mi),2.2582
10,Wind_Speed(mph),7.3974
11,Precipitation(in),28.5232
12,Weather_Condition,2.2202
25,Sunrise_Sunset,0.2966


<p> Missing percentage </p>

In [40]:
# for the Precipitation, it will affect the model so Precipitation_NA will be a flag for missing values   
df['Precipitation_NA'] = 0
df.loc[df['Precipitation(in)'].isnull(),'Precipitation_NA'] = 1
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(df['Precipitation(in)'].median())
df.loc[:5,['Precipitation(in)','Precipitation_NA']]
df['Precipitation_NA'].value_counts() 

Precipitation_NA
0    357384
1    142616
Name: count, dtype: int64

<p> Create a flag to indicate which values have been filled (as median) in the (Precipitation(in)) feature </p>

In [43]:
df = df.dropna(subset=['Street','Airport_Code','Sunrise_Sunset'], axis = 0)
print("The shape of data is:",(df.shape))


The shape of data is: (496544, 29)


<p>Drop rows that contain N/A for these features (note that: small percentages were missing)</p>

In [45]:
df['Month'] = df['Start_Time'].dt.month


df['Hour'] = df['Start_Time'].dt.hour


df.loc[:4,['Start_Time','Hour', 'Month']]

,Start_Time,Hour,Month
0,2019-06-12 10:10:56,10,6
1,2022-12-03 23:37:14,23,12
2,2022-08-20 13:13:00,13,8
3,2022-02-21 17:43:04,17,2
4,2020-12-04 01:46:00,1,12


<p> Feature extraction</p>

In [46]:
Weather_data=['Temperature(F)','Humidity(%)','Pressure(in)','Visibility(mi)','Wind_Speed(mph)']
print("The number of remaining missing values: ")
for i in Weather_data:
  df[i] = df.groupby(['Airport_Code','Month'])[i].transform(lambda x: x.fillna(x.median()))
  print( i + " : " + df[i].isnull().sum().astype(str))

The number of remaining missing values: 
Temperature(F) : 1439
Humidity(%) : 1447
Pressure(in) : 1443
Visibility(mi) : 2902
Wind_Speed(mph) : 2980


<p>Fill missing data with median, grouped by ['Airport_Code','Month'] (location, time)</p>

In [47]:
df.dropna(subset= Weather_data, axis=0, inplace=True)
print("The shape of data is:",(df.shape))

The shape of data is: (493392, 29)


In [48]:
print("Wind Conditions: ", df['Weather_Condition'].unique())


Wind Conditions:  <StringArray>
[                        'Fair',                   'Wintry Mix',
                   'Light Rain',                       'Cloudy',
                'Mostly Cloudy',                'Partly Cloudy',
                        'Clear',             'Scattered Clouds',
                          'Fog',                     'Overcast',
 ...
    'Light Rain Shower / Windy', 'Light Thunderstorms and Snow',
            'Light Snow Grains',             'Thunder and Hail',
        'Drifting Snow / Windy',                 'Volcanic Ash',
                 'Mist / Windy',           'Light Blowing Snow',
            'Low Drifting Snow',                         'Sand']
Length: 109, dtype: str


<p>Get idea of 'Weather_Condition' values  </p>

In [49]:


def map_weather(val):
    # 1. Handle Nulls/Empty: Leave exactly as they are
    if pd.isna(val) or str(val).strip() == '' or val == 'N/A Precipitation':
        return None
    
    val = str(val).lower()

    # 2. Snow (Highest Priority)
    if any(x in val for x in ['snow', 'sleet', 'ice', 'hail', 'wintry', 'grains']):
        return 'snow'
    
    # 3. Rain
    if any(x in val for x in ['rain', 'drizzle', 'shower', 'storm', 'thunder', 'squall']):
        return 'rain'
    
    # 4. Fog
    if any(x in val for x in ['fog', 'mist', 'haze', 'smoke', 'dust', 'sand', 'ash', 'whirlwind']):
        return 'fog'
    
    # 5. Cloudy
    if any(x in val for x in ['cloud', 'overcast']):
        return 'cloudy'
    
    # 6. Everything Else -> Clear
    # This captures 'Clear', 'Fair', 'Windy', 'Heavy ', 'Light ', etc.
    return 'clear'

# Application
df['Weather_Condition'] = df['Weather_Condition'].apply(map_weather)

print(df['Weather_Condition'].value_counts())
print(df['Weather_Condition'].isnull().sum())

Weather_Condition
clear     219682
cloudy    203630
rain       39005
fog        13131
snow       11018
Name: count, dtype: int64
6926


<p>Simplify Weather Categories</p>

In [50]:
print("The number of remaining missing values: ")
df['Weather_Condition'] = df.groupby(['Airport_Code', 'Month'])['Weather_Condition'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))
print( "Weather Condition"+ " : " + df['Weather_Condition'].isnull().sum().astype(str)) 

The number of remaining missing values: 
Weather Condition : 62


<p>Fill missing data with mode, grouped by ['Airport_Code','Month'] (location, time)</p>

In [51]:
df.dropna(subset=['Weather_Condition'], axis=0, inplace=True)
df.shape

(493330, 29)

In [52]:
missing = pd.DataFrame(df.isnull().sum()).reset_index()
missing.columns = ['Feature', 'Missing_Percent(%)']
missing['Missing_Percent(%)'] = missing['Missing_Percent(%)'].transform(lambda x: x / df.shape[0] * 100)
missing.loc[missing['Missing_Percent(%)']>=0,:]

,Feature,Missing_Percent(%)
0,Severity,0.0
1,Start_Time,0.0
2,Start_Lat,0.0
3,Start_Lng,0.0
4,Street,0.0
5,Airport_Code,0.0
6,Temperature(F),0.0
7,Humidity(%),0.0
8,Pressure(in),0.0
9,Visibility(mi),0.0


<p>Last check</p>